# 02a: NASA KOI Data Audit

Скачивание, разведка и очистка табличных данных NASA Exoplanet Archive (Kepler Objects of Interest).

**Цель** - получить чистый датасет без утечек (data leakage), пригодный для обучения табличной ветви ансамбля.

**Выходные файлы:**
- `data/processed_physical_signs/koi_cumulative.csv` - полный «сырой» датасет (все скачанные колонки)
- `data/physical_signs_final/koi_clean.csv` - очищенный бинарный датасет (CONFIRMED vs FALSE POSITIVE)
- `data/physical_signs_final/koi_candidates.csv` - строки CANDIDATE (отложены для будущего)
- `data/physical_signs_final/koi_keys.csv` - kepid/kepoi_name/kepler_name для трассировки

### Импотры и пути

In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
from pathlib import Path
from IPython.display import display

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "processed_physical_signs"
FINAL_DIR = DATA_DIR / "physical_signs_final"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("FINAL_DIR:", FINAL_DIR.resolve())

RAW_DIR: C:\Users\Vladislav\.vscode\ML\exoplanet-detector\data\processed_physical_signs
FINAL_DIR: C:\Users\Vladislav\.vscode\ML\exoplanet-detector\data\physical_signs_final


## Скачивание данных

Расширенный набор колонок по сравнению с предыдущей версией. В список попадают:

- **Идентификаторы и таргет**: `kepid`, `kepoi_name`, `kepler_name`, `koi_disposition`.
- **Физические признаки**: транзитные (`koi_period`, `koi_time0bk`, `koi_impact`, `koi_duration`, `koi_depth`, `koi_prad`, `koi_teq`, `koi_insol`, `koi_ror`, `koi_dor`), звёздные (`koi_steff`, `koi_slogg`, `koi_srad`, `koi_smass`, `koi_smet`, `koi_sage`, `koi_srho`), наблюдательные (`koi_kepmag`, `koi_model_snr`, `koi_num_transits`, `koi_max_mult_ev`, `koi_max_sngle_ev`, `koi_quarters`, `koi_count`).
- **Данные, зафиксированные как утечки**: `koi_pdisposition`, `koi_score`, `koi_fpflag_*`.
- **Ошибки измерений** (`*_err1`, `*_err2`) - понадобятся позже.

In [2]:
desired_columns = [
    # Идентификаторы и таргет
    "kepid", "kepoi_name", "kepler_name", "koi_disposition",
    # Утечки для аудита
    "koi_pdisposition", "koi_score",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec",
    # Транзитные
    "koi_period", "koi_time0bk", "koi_impact", "koi_duration",
    "koi_depth", "koi_prad", "koi_teq", "koi_insol",
    "koi_ror", "koi_dor",
    # Звёздные
    "koi_steff", "koi_slogg", "koi_srad", "koi_smass",
    "koi_smet", "koi_sage", "koi_srho",
    # Наблюдательные
    "koi_kepmag", "koi_model_snr", "koi_num_transits",
    "koi_max_mult_ev", "koi_max_sngle_ev",
    "koi_quarters", "koi_count",
    # Ошибки
    "koi_period_err1", "koi_period_err2",
    "koi_time0bk_err1", "koi_time0bk_err2",
    "koi_impact_err1", "koi_impact_err2",
    "koi_duration_err1", "koi_duration_err2",
    "koi_depth_err1", "koi_depth_err2",
    "koi_prad_err1", "koi_prad_err2",
    "koi_teq_err1", "koi_teq_err2",
    "koi_insol_err1", "koi_insol_err2",
    "koi_ror_err1", "koi_ror_err2",
    "koi_dor_err1", "koi_dor_err2",
    "koi_steff_err1", "koi_steff_err2",
    "koi_slogg_err1", "koi_slogg_err2",
    "koi_srad_err1", "koi_srad_err2",
    "koi_smass_err1", "koi_smass_err2",
    "koi_smet_err1", "koi_smet_err2",
    "koi_sage_err1", "koi_sage_err2",
    "koi_srho_err1", "koi_srho_err2",
    "koi_kepmag_err",
]

fetch_columns = [c for c in desired_columns if c]
print(f"Скачивание {len(fetch_columns)} колонок")

query = f"SELECT {','.join(fetch_columns)} FROM cumulative"
response = requests.get(
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync",
    params={"query": query, "format": "csv"},
    timeout=180,
)
response.raise_for_status()

koi = pd.read_csv(StringIO(response.text))
koi.to_csv(RAW_DIR / "koi_cumulative.csv", index=False)

print(f"Скачано: {koi.shape[0]} строк × {koi.shape[1]} колонок")
print(f"Сохранено в {RAW_DIR / 'koi_cumulative.csv'}")

Скачивание 69 колонок
Скачано: 9564 строк × 69 колонок
Сохранено в ..\data\processed_physical_signs\koi_cumulative.csv


### Структура данных

In [3]:
print("Форма:", koi.shape)
print("\nРаспределение koi_disposition:")
display(koi["koi_disposition"].value_counts())
print("\nТипы данных (основные):")
print(koi.dtypes.head(15))

Форма: (9564, 69)

Распределение koi_disposition:


koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64


Типы данных (основные):
kepid                 int64
kepoi_name           object
kepler_name          object
koi_disposition      object
koi_pdisposition     object
koi_score           float64
koi_fpflag_nt         int64
koi_fpflag_ss         int64
koi_fpflag_co         int64
koi_fpflag_ec         int64
koi_period          float64
koi_time0bk         float64
koi_impact          float64
koi_duration        float64
koi_depth           float64
dtype: object


In [4]:
koi.head(5)

,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,...,koi_srad_err2,koi_smass_err1,koi_smass_err2,koi_smet_err1,koi_smet_err2,koi_sage_err1,koi_sage_err2,koi_srho_err1,koi_srho_err2,koi_kepmag_err
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,-0.061,0.052,-0.046,0.15,-0.15,NaN,NaN,0.33173,-1.09986,NaN
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,0,...,-0.061,0.052,-0.046,0.15,-0.15,NaN,NaN,2.20489,-2.49638,NaN
2,10811496,K00753.01,NaN,CANDIDATE,CANDIDATE,0.000,0,0,0,0,...,-0.078,0.110,-0.121,0.30,-0.30,NaN,NaN,35.03293,-2.75453,NaN
3,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,...,-0.067,0.093,-0.077,0.30,-0.30,NaN,NaN,0.00917,-0.01837,NaN
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,...,-0.133,0.151,-0.136,0.25,-0.30,NaN,NaN,2.71141,-1.74541,NaN


### Пропуски

In [5]:
na_counts = koi.isna().sum().sort_values(ascending=False)
na_pct = (na_counts / len(koi) * 100).round(2)
na_table = pd.DataFrame({"n_nan": na_counts, "pct": na_pct})
display(na_table[na_table["n_nan"] > 0])

,n_nan,pct
koi_sage_err2,9564,100.00
koi_sage_err1,9564,100.00
koi_teq_err1,9564,100.00
koi_kepmag_err,9564,100.00
koi_sage,9564,100.00
koi_teq_err2,9564,100.00
kepler_name,6816,71.27
koi_score,1510,15.79
koi_max_sngle_ev,1142,11.94
koi_num_transits,1142,11.94


*kepler_name на 71% состоит из NaN - это нормально, особенность данных (кеплеровские имена дают только подтверждённым планетам, что также является и неявной утечкой, так что от этого столба следует избавиться полностью - будет удалён на этапе с сохранением ключей)*
*6 столбцов на 100% состоят из NaN*

**Решение: утилизировать**

In [6]:
koi.drop(columns=["koi_sage_err2", "koi_sage_err1", "koi_teq_err1", "koi_kepmag_err", "koi_sage", "koi_teq_err2"], inplace=True)

### Опасность: утечки через флаги

> **Этот раздел - исследовательская часть с прошлой версии, оставленная без изменений.**

Из 2748 строк с положительным `koi_disposition` всего 18 строк имеют хоть какой-то из флагов, а из 4839 с отрицательным - всего 95 не имеют флагов вообще. При первом запуске моделей на датасете с этими флагами результаты были подозрительно хороши (XGB с recall 99.4% на кросс-валидации). Наличие любого флага слишком сильно коррелирует с правильным `koi_disposition`. Убираем флаги полностью, чтобы сделать результаты более справедливыми.

In [7]:
koi_binary = koi[koi["koi_disposition"] != "CANDIDATE"].drop(columns=["koi_pdisposition", "kepid", "kepoi_name", "kepler_name"]).copy()
koi_binary["koi_disposition"] = koi_binary["koi_disposition"].map({"CONFIRMED": 1, "FALSE POSITIVE": 0})
corr_matrix = koi_binary.corr()
display(corr_matrix["koi_disposition"].sort_values(ascending=False))

koi_binary_zero = koi_binary[koi_binary["koi_disposition"] == 0].copy()
koi_binary_one = koi_binary[koi_binary["koi_disposition"] == 1].copy()
display(koi_binary_one)
display(koi_binary_zero)

one_with_flags = koi_binary["koi_disposition"][(koi_binary["koi_disposition"] == 1) & ((koi_binary["koi_fpflag_nt"] + koi_binary["koi_fpflag_ss"] + koi_binary["koi_fpflag_co"] + koi_binary["koi_fpflag_ec"]) > 0)].count()
print(one_with_flags)
zero_without_flags = koi_binary["koi_disposition"][(koi_binary["koi_disposition"] == 0) & ((koi_binary["koi_fpflag_nt"] + koi_binary["koi_fpflag_ss"] + koi_binary["koi_fpflag_co"] + koi_binary["koi_fpflag_ec"]) == 0)].count()
print(zero_without_flags)

koi_disposition      1.000000
koi_score            0.946880
koi_count            0.483494
koi_smet_err2        0.458231
koi_steff_err2       0.385350
koi_smet             0.329378
koi_slogg_err2       0.303678
koi_smass_err2       0.240140
koi_duration_err2    0.205408
koi_slogg            0.181666
koi_period_err2      0.152331
koi_quarters         0.150087
koi_time0bk_err2     0.139684
koi_dor_err2         0.126767
koi_srad_err2        0.085163
koi_srho_err2        0.075913
koi_ror_err2         0.056927
koi_kepmag           0.056550
koi_insol_err2       0.037334
koi_prad_err2        0.020619
koi_depth_err2       0.020312
koi_impact_err2      0.003433
koi_fpflag_nt       -0.013029
koi_depth_err1      -0.020312
koi_prad            -0.024657
koi_prad_err1       -0.034531
koi_insol           -0.039214
koi_insol_err1      -0.054002
koi_time0bk         -0.063341
koi_ror             -0.065609
koi_srho            -0.067368
koi_impact          -0.076322
koi_srad            -0.085304
koi_dor   

,koi_disposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_time0bk,koi_impact,koi_duration,...,koi_slogg_err1,koi_slogg_err2,koi_srad_err1,koi_srad_err2,koi_smass_err1,koi_smass_err2,koi_smet_err1,koi_smet_err2,koi_srho_err1,koi_srho_err2
0,1,1.000,0,0,0,0,9.488036,170.53875,0.146,2.9575,...,0.064,-0.096,0.105,-0.061,0.052,-0.046,0.15,-0.15,0.33173,-1.09986
1,1,0.969,0,0,0,0,54.418383,162.51384,0.586,4.5070,...,0.064,-0.096,0.105,-0.061,0.052,-0.046,0.15,-0.15,2.20489,-2.49638
4,1,1.000,0,0,0,0,2.525592,171.59555,0.701,1.6545,...,0.070,-0.210,0.334,-0.133,0.151,-0.136,0.25,-0.30,2.71141,-1.74541
5,1,1.000,0,0,0,0,11.094321,171.20116,0.538,4.5945,...,0.054,-0.229,0.315,-0.105,0.135,-0.150,0.25,-0.30,0.33286,-0.38858
6,1,1.000,0,0,0,0,4.134435,172.97937,0.762,3.1402,...,0.054,-0.229,0.315,-0.105,0.135,-0.150,0.25,-0.30,0.74768,-0.26357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8817,1,1.000,0,0,0,0,4.485592,135.43172,0.442,0.8161,...,0.055,-0.050,0.034,-0.025,0.038,-0.025,0.10,-0.10,32.28736,-34.98688
8956,1,NaN,0,0,0,0,8.152759,134.19046,0.461,1.7460,...,0.140,-0.140,0.125,-0.054,0.148,-0.049,0.36,-0.24,15.06731,-12.50699
9014,1,0.771,0,0,0,0,384.847556,314.97000,0.059,9.9690,...,0.034,-0.127,0.150,-0.075,0.082,-0.109,0.30,-0.30,1.69325,-1.18690
9083,1,0.000,0,0,0,0,3.875943,134.84758,0.025,2.3140,...,0.013,-0.117,0.125,-0.033,0.043,-0.073,0.15,-0.15,0.05891,-1.90538


,koi_disposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_time0bk,koi_impact,koi_duration,...,koi_slogg_err1,koi_slogg_err2,koi_srad_err1,koi_srad_err2,koi_smass_err1,koi_smass_err2,koi_smet_err1,koi_smet_err2,koi_srho_err1,koi_srho_err2
3,0,0.000,0,1,0,0,1.736952,170.307565,1.276,2.40641,...,0.053,-0.168,0.201,-0.067,0.093,-0.077,0.30,-0.30,0.00917,-0.01837
8,0,0.000,0,1,1,0,7.361790,132.250530,1.169,5.02200,...,0.182,-0.098,0.322,-0.483,0.121,-0.207,0.15,-0.15,0.27864,-0.00053
14,0,0.000,0,1,0,0,11.521446,170.839688,2.483,3.63990,...,0.033,-0.176,0.224,-0.075,0.098,-0.109,0.25,-0.30,2.30242,-0.41065
15,0,0.000,0,1,0,0,19.403938,172.484253,0.804,12.21550,...,0.072,-0.048,0.072,-0.065,0.085,-0.036,0.35,-0.30,0.00107,-0.00172
16,0,0.000,0,1,0,0,19.221389,184.552164,1.065,4.79843,...,0.052,-0.208,0.308,-0.096,0.139,-0.139,0.25,-0.35,0.00657,-0.02917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9557,0,0.008,1,0,0,0,373.893980,261.496800,0.963,27.66000,...,0.078,-0.052,0.074,-0.066,0.082,-0.032,0.35,-0.30,0.28310,-0.00537
9558,0,0.000,0,0,0,1,8.589871,132.016100,0.765,4.80600,...,0.231,-0.189,0.313,-0.228,0.125,-0.073,0.30,-0.25,0.87418,-0.07724
9559,0,0.000,0,1,1,0,0.527699,131.705093,1.252,3.22210,...,0.035,-0.196,0.237,-0.079,0.084,-0.115,0.20,-0.30,0.01693,-0.01718
9561,0,0.021,0,0,1,0,0.681402,132.181750,0.147,0.86500,...,0.056,-0.224,0.341,-0.114,0.146,-0.162,0.25,-0.35,23.11894,-8.63287


18
95


**Принято решение:** *полностью удалить колонки флагов из датасета*

In [8]:
koi.drop(columns=["koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"], inplace=True)

## Дополнительный аудит утечек

### `koi_score` - скор уверенности Robovetter

`koi_score` - это выходной скор того же пайплайна веттинга, который сгенерировал `koi_disposition` (извсетсная утечка с прошлой версии ноутбука). Если его оставить, модель будет просто повторять решение NASA. Стоит удалить оба столбца.

In [9]:
koi.drop(columns=["koi_score", "koi_pdisposition"], inplace=True)
print("После удаления утечек и неинформативных столбцов:", koi.shape)
print("Осталось колонок:", list(koi.columns))

После удаления утечек и неинформативных столбцов: (9564, 57)
Осталось колонок: ['kepid', 'kepoi_name', 'kepler_name', 'koi_disposition', 'koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_ror', 'koi_dor', 'koi_steff', 'koi_slogg', 'koi_srad', 'koi_smass', 'koi_smet', 'koi_srho', 'koi_kepmag', 'koi_model_snr', 'koi_num_transits', 'koi_max_mult_ev', 'koi_max_sngle_ev', 'koi_quarters', 'koi_count', 'koi_period_err1', 'koi_period_err2', 'koi_time0bk_err1', 'koi_time0bk_err2', 'koi_impact_err1', 'koi_impact_err2', 'koi_duration_err1', 'koi_duration_err2', 'koi_depth_err1', 'koi_depth_err2', 'koi_prad_err1', 'koi_prad_err2', 'koi_insol_err1', 'koi_insol_err2', 'koi_ror_err1', 'koi_ror_err2', 'koi_dor_err1', 'koi_dor_err2', 'koi_steff_err1', 'koi_steff_err2', 'koi_slogg_err1', 'koi_slogg_err2', 'koi_srad_err1', 'koi_srad_err2', 'koi_smass_err1', 'koi_smass_err2', 'koi_smet_err1', 'koi_smet_err2', 'koi_srho_err1', 'koi_srho_err2']


## Обработка строковых и производных колонок

`koi_quarters` - строка вида `"Q0,Q1,Q2,..."`. Можно преобразовать в числовой признак `koi_n_quarters` (количество кварталов наблюдения), а исходную строку удалить.

In [10]:
if "koi_quarters" in koi.columns:
    koi["koi_n_quarters"] = koi["koi_quarters"].fillna("").apply(
        lambda s: len([q for q in str(s).split(",") if q.strip()]) if s else 0
    )
    koi.drop(columns=["koi_quarters"], inplace=True)
    print("koi_quarters -> koi_n_quarters создан")
else:
    print("koi_quarters отсутствует, пропуск")

print("Текущая форма:", koi.shape)

koi_quarters -> koi_n_quarters создан
Текущая форма: (9564, 57)


## Разделение на CANDIDATE и бинарный таргет

Строки `CANDIDATE` - это случаи, где модель NASA не приняла окончательного решения. Они не подходят для обучения бинарного классификатора (CONFIRMED vs FALSE POSITIVE) и могут запутать модель. Но могут пригодиться в последующей работе, поэтому стоит отложить их в отдельный файл.

Перед разделением - фиксация `kepid`/`kepoi_name`/`kepler_name` в отдельный датафрейм `keys`. Может понадобиться для дальшейшей синхронизации ансамбля.

In [11]:
keys = koi[["kepid", "kepoi_name", "kepler_name"]].copy()
koi.drop(columns=["kepler_name"], inplace=True)

koi_candidates = koi[koi["koi_disposition"] == "CANDIDATE"].copy()
koi_binary = koi[koi["koi_disposition"] != "CANDIDATE"].copy()

print("CANDIDATE:", koi_candidates.shape)
print("Бинарный таргет:", koi_binary.shape)
print("\nРаспределение:")
print(koi_binary["koi_disposition"].value_counts())

CANDIDATE: (1977, 56)
Бинарный таргет: (7587, 56)

Распределение:
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
Name: count, dtype: int64


### Обработка пропусков

In [12]:
# Удаление только из критичных колонок, в обычных можно пока оставить
critical_cols = [
    "koi_period", "koi_duration", "koi_depth", "koi_prad",
    "koi_teq", "koi_insol", "koi_model_snr",
    "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag",
]

before = len(koi_binary)
koi_binary = koi_binary.dropna(subset=critical_cols)
print(f"Удалено строк с NaN в критичных колонках: {before - len(koi_binary)}")
print(f"Осталось: {len(koi_binary)}")

Удалено строк с NaN в критичных колонках: 260
Осталось: 7327


### Удаление полных дубликатов

In [13]:
before = len(koi_binary)
koi_binary = koi_binary.drop_duplicates()
print(f"Удалено дубликатов: {before - len(koi_binary)}")
print(f"Итоговая форма: {koi_binary.shape}")

Удалено дубликатов: 0
Итоговая форма: (7327, 56)


## Группировка по `kepid` для GroupKFold

Важный момент: у одной звезды (`kepid`) может быть **несколько** KOI-кандидатов. При обычном `train_test_split` KOI одной и той же звезды попадут и в train, и в test - модель будет видеть звезду на обоих этапах - утечка.

Поэтому для кросс-валидации нужно использовать **GroupKFold** по `kepid`.

 *Проверка, что групп больше одной и что в каждой группе достаточно примеров:*

In [14]:
group_sizes = koi_binary.groupby("kepid").size()
print(f"Уникальных kepid: {len(group_sizes)}")
print(f"Звёзд с >1 KOI: {(group_sizes > 1).sum()}")
print(f"Максимум KOI у одной звезды: {group_sizes.max()}")
print(f"Медиана KOI на звезду: {group_sizes.median()}")

Уникальных kepid: 6423
Звёзд с >1 KOI: 614
Максимум KOI у одной звезды: 7
Медиана KOI на звезду: 1.0


## Сохранение финального датасета

Преобразование таргета в бинарный (0/1) и затем сохранение:
- `koi_clean.csv` - чистый датасет (kepid + kepoi_name + признаки + label)
- `koi_candidates.csv` - отложенные CANDIDATE
- `koi_keys.csv` - kepid/kepoi_name/kepler_name для трассировки

In [15]:
# Бинарный таргет
koi_binary["label"] = koi_binary["koi_disposition"].map({"CONFIRMED": 1, "FALSE POSITIVE": 0})
koi_binary.drop(columns=["koi_disposition"], inplace=True)

# Финальный порядок колонок: kepid, kepoi_name, признаки, label
front_cols = ["kepid", "kepoi_name"]
label_col = ["label"]
feature_cols = [c for c in koi_binary.columns if c not in front_cols + label_col]
koi_final = koi_binary[front_cols + feature_cols + label_col].copy()

# Проверка: никаких утечек
leak_check = ["koi_score", "koi_pdisposition", "kepler_name",
              "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]
leaks_present = [c for c in leak_check if c in koi_final.columns]
assert not leaks_present, f"Утечки не удалены: {leaks_present}"
print("Проверка на утечки пройдена")

# Сохранение
koi_final.to_csv(FINAL_DIR / "koi_clean.csv", index=False)
koi_candidates.drop(columns=["koi_disposition"]).to_csv(FINAL_DIR / "koi_candidates.csv", index=False)
keys.to_csv(FINAL_DIR / "koi_keys.csv", index=False)

print(f"\nkoi_clean.csv: {koi_final.shape}")
print(f"koi_candidates.csv: {koi_candidates.shape}")
print(f"koi_keys.csv: {keys.shape}")
print(f"\nКлассы в финальном датасете:")
print(koi_final["label"].value_counts())

Проверка на утечки пройдена

koi_clean.csv: (7327, 56)
koi_candidates.csv: (1977, 56)
koi_keys.csv: (9564, 3)

Классы в финальном датасете:
label
0    4581
1    2746
Name: count, dtype: int64
